# Chapter 8: Advanced Topics and Future Horizons

## Complete Code Implementations

This notebook contains all code listings from Chapter 8, covering advanced AI agent topics including:

- **8.1** Persistent Agent with Identity Preservation
- **8.2** Distributed Multi-Agent Coordination System
- **8.3** Neurosymbolic Reasoning Agent
- **8.4** Agent Marketplace Registration and Discovery System

---

## Setup and Dependencies

Install required packages before running the examples.

In [ ]:
# Install dependencies
# !pip install langgraph langchain langchain-openai pydantic numpy

In [ ]:
# Common imports used across listings
from typing import TypedDict, Optional, List, Dict, Any, Literal, Annotated
from datetime import datetime
from enum import Enum
from dataclasses import dataclass, field
from pydantic import BaseModel, Field
import json
import uuid
import hashlib
import time
import random

---

## Listing 8-1: Persistent Agent with Identity Preservation

A persistent agent architecture that maintains identity across interactions, accumulates episodic memories with temporal decay, and periodically self-reflects on performance. Uses MemorySaver for checkpoint persistence.

In [ ]:
"""
Listing 8-1: Persistent Agent with Identity Preservation

Demonstrates a persistent agent that maintains identity via an immutable
hash, accumulates episodic memories with importance-weighted decay,
and self-reflects every N interactions. All LLM calls are simulated.
"""

from typing import TypedDict, Annotated, List, Dict, Any
from dataclasses import dataclass, field
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import json, hashlib, time, uuid, random


# ------------------------------------------------------------------ #
# Data Classes                                                        #
# ------------------------------------------------------------------ #

@dataclass
class AgentIdentity:
    """Core agent identity anchored to an immutable hash."""
    agent_id: str
    name: str
    created_at: float
    version: str = "1.0"
    core_values: list = field(default_factory=lambda: [
        "accuracy", "helpfulness", "safety", "transparency",
    ])
    behavioral_constraints: list = field(default_factory=lambda: [
        "never fabricate data",
        "acknowledge uncertainty",
        "defer to human judgment on ethical questions",
        "maintain audit trail for all decisions",
    ])
    specialization: str = "general research assistant"

    def identity_hash(self) -> str:
        """Compute immutable identity hash from core values and constraints."""
        identity_str = json.dumps({
            "id": self.agent_id,
            "values": sorted(self.core_values),
            "constraints": sorted(self.behavioral_constraints),
        }, sort_keys=True)
        return hashlib.sha256(identity_str.encode()).hexdigest()[:16]

    def to_dict(self) -> dict:
        return {
            "agent_id": self.agent_id,
            "name": self.name,
            "created_at": self.created_at,
            "version": self.version,
            "core_values": self.core_values,
            "behavioral_constraints": self.behavioral_constraints,
            "specialization": self.specialization,
            "identity_hash": self.identity_hash(),
        }


@dataclass
class EpisodicMemory:
    """A single episodic memory with importance-weighted temporal decay."""
    memory_id: str
    timestamp: float
    context: str
    outcome: str
    lesson: str
    importance: float = 0.5
    access_count: int = 0

    def relevance_score(self, current_time: float, query: str = "") -> float:
        """Score combining recency, importance, and access frequency."""
        elapsed_days = (current_time - self.timestamp) / 86400
        half_life = 30.0 * max(self.importance, 0.1)
        recency = 0.5 ** (elapsed_days / half_life)
        access_boost = min(self.access_count * 0.05, 0.3)
        # Simple keyword overlap for query relevance
        query_words = set(query.lower().split()) if query else set()
        context_words = set(self.context.lower().split())
        keyword_score = len(query_words & context_words) / max(len(query_words), 1) if query_words else 0
        return min(recency * self.importance + access_boost + keyword_score * 0.2, 1.0)

    def to_dict(self) -> dict:
        return {
            "memory_id": self.memory_id,
            "timestamp": self.timestamp,
            "context": self.context,
            "outcome": self.outcome,
            "lesson": self.lesson,
            "importance": self.importance,
            "access_count": self.access_count,
        }


# ------------------------------------------------------------------ #
# State                                                               #
# ------------------------------------------------------------------ #

class PersistentAgentState(TypedDict):
    """State for persistent agent with identity and memory."""
    messages: Annotated[list[BaseMessage], add_messages]
    identity: dict
    memories: list[dict]
    interaction_count: int
    performance_log: list[dict]
    self_evaluation: dict


# ------------------------------------------------------------------ #
# Helper Functions                                                    #
# ------------------------------------------------------------------ #

def retrieve_relevant_memories(memories: list[dict], query: str,
                                current_time: float, top_k: int = 3) -> list[dict]:
    """Retrieve top-k most relevant memories for the current query."""
    scored = []
    for m_dict in memories:
        m = EpisodicMemory(**m_dict)
        score = m.relevance_score(current_time, query)
        m.access_count += 1
        scored.append((score, m.to_dict()))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [m for _, m in scored[:top_k]]


# ------------------------------------------------------------------ #
# Node Functions                                                      #
# ------------------------------------------------------------------ #

def process_interaction(state: PersistentAgentState) -> dict:
    """Process the current interaction using identity and relevant memories."""
    identity = state["identity"]
    current_msg = state["messages"][-1].content if state["messages"] else ""
    current_time = time.time()

    # Retrieve relevant memories
    relevant = retrieve_relevant_memories(
        state.get("memories", []), current_msg, current_time
    )

    # Build memory-aware context (simulated LLM call)
    memory_context = ""
    if relevant:
        memory_context = "\n\nRelevant past experiences:\n"
        for m in relevant:
            memory_context += f"- {m['context']}: {m['lesson']}\n"

    response = (
        f"[{identity['name']}] Processing request as {identity['specialization']}. "
        f"Identity hash: {identity['identity_hash']}. "
        f"Drawing on {len(relevant)} relevant memories."
        f"{memory_context}"
        f"\n\nResponse to: '{current_msg[:80]}...'" if len(current_msg) > 80 else
        f"\n\nResponse to: '{current_msg}'"
    )

    return {
        "messages": [AIMessage(content=response)],
        "interaction_count": state.get("interaction_count", 0) + 1,
    }


def record_episode(state: PersistentAgentState) -> dict:
    """Record the current interaction as an episodic memory."""
    current_msg = ""
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            current_msg = msg.content
            break

    # Simulated lesson extraction (would use LLM in production)
    lessons = [
        "Complex queries benefit from step-by-step decomposition",
        "Users appreciate concrete examples over abstract explanations",
        "When uncertain, explicitly state confidence level",
        "Cross-referencing multiple sources improves accuracy",
        "Domain-specific terminology should be defined on first use",
    ]

    new_memory = EpisodicMemory(
        memory_id=str(uuid.uuid4())[:8],
        timestamp=time.time(),
        context=current_msg[:100] if current_msg else "interaction",
        outcome="completed",
        lesson=random.choice(lessons),
        importance=round(random.uniform(0.3, 0.9), 2),
    )

    memories = list(state.get("memories", []))
    memories.append(new_memory.to_dict())

    return {"memories": memories}


def self_reflect(state: PersistentAgentState) -> dict:
    """Periodic self-evaluation against core values."""
    identity = state["identity"]
    interaction_count = state.get("interaction_count", 0)

    # Simulated self-evaluation
    evaluation = {
        "timestamp": time.time(),
        "interactions_evaluated": interaction_count,
        "value_alignment": {
            value: round(random.uniform(0.8, 1.0), 2)
            for value in identity.get("core_values", [])
        },
        "constraint_compliance": {
            constraint: True
            for constraint in identity.get("behavioral_constraints", [])
        },
        "memory_count": len(state.get("memories", [])),
        "improvement_suggestions": [
            "Increase use of structured reasoning for complex queries",
            "Improve uncertainty communication in borderline cases",
        ],
        "overall_alignment_score": round(random.uniform(0.85, 0.98), 3),
    }

    log = list(state.get("performance_log", []))
    log.append({"type": "self_reflection", "timestamp": time.time(),
                "alignment_score": evaluation["overall_alignment_score"]})

    return {
        "self_evaluation": evaluation,
        "performance_log": log,
        "messages": [AIMessage(content=(
            f"Self-reflection complete. Alignment score: "
            f"{evaluation['overall_alignment_score']}. "
            f"Memories accumulated: {evaluation['memory_count']}."
        ))],
    }


def should_reflect(state: PersistentAgentState) -> str:
    """Decide whether to self-reflect (every 5 interactions)."""
    count = state.get("interaction_count", 0)
    if count > 0 and count % 5 == 0:
        return "reflect"
    return "continue"


# ------------------------------------------------------------------ #
# Graph Construction                                                  #
# ------------------------------------------------------------------ #

def build_persistent_agent():
    """Build persistent agent with memory and self-reflection."""
    workflow = StateGraph(PersistentAgentState)
    workflow.add_node("process", process_interaction)
    workflow.add_node("record", record_episode)
    workflow.add_node("reflect", self_reflect)

    workflow.set_entry_point("process")
    workflow.add_edge("process", "record")
    workflow.add_conditional_edges(
        "record",
        should_reflect,
        {"reflect": "reflect", "continue": END},
    )
    workflow.add_edge("reflect", END)

    return workflow.compile(checkpointer=MemorySaver())


# ------------------------------------------------------------------ #
# Usage Example                                                       #
# ------------------------------------------------------------------ #

# Create agent identity
identity = AgentIdentity(
    agent_id="agent-001",
    name="ResearchBot",
    created_at=time.time(),
    specialization="scientific research assistant",
)

graph = build_persistent_agent()
config = {"configurable": {"thread_id": "persistent-session-1"}}

# Simulate multi-turn conversation showing memory accumulation
conversations = [
    "What are the latest advances in perovskite solar cells?",
    "How does the bandgap of CsPbI3 compare to silicon?",
    "Can you explain the degradation mechanisms in hybrid perovskites?",
    "What synthesis methods are used for lead-free perovskites?",
    "Summarize the key challenges in perovskite commercialization",
]

print("=" * 70)
print("PERSISTENT AGENT DEMONSTRATION")
print("=" * 70)
print(f"\nAgent: {identity.name} (ID: {identity.agent_id})")
print(f"Identity hash: {identity.identity_hash()}")
print(f"Specialization: {identity.specialization}")

state = {
    "messages": [],
    "identity": identity.to_dict(),
    "memories": [],
    "interaction_count": 0,
    "performance_log": [],
    "self_evaluation": {},
}

for i, msg in enumerate(conversations):
    print(f"\n{'─' * 50}")
    print(f"Turn {i+1}: {msg}")
    state["messages"] = [HumanMessage(content=msg)]
    result = graph.invoke(state, config)
    state["memories"] = result.get("memories", state["memories"])
    state["interaction_count"] = result.get("interaction_count", state["interaction_count"])
    state["performance_log"] = result.get("performance_log", state["performance_log"])
    state["self_evaluation"] = result.get("self_evaluation", state["self_evaluation"])

    # Print last AI message
    for m in reversed(result["messages"]):
        if isinstance(m, AIMessage):
            print(f"Agent: {m.content[:200]}")
            break

    print(f"Memories: {len(state['memories'])} | Interactions: {state['interaction_count']}")

if state.get("self_evaluation"):
    print(f"\n{'─' * 50}")
    print("Self-evaluation results:")
    ev = state["self_evaluation"]
    print(f"  Alignment score: {ev.get('overall_alignment_score', 'N/A')}")
    print(f"  Value alignment: {json.dumps(ev.get('value_alignment', {}), indent=4)}")


---

## Listing 8-2: Distributed Multi-Agent Coordination System

A distributed coordination system with a service registry for agent discovery, a message bus for inter-agent communication, task decomposition, parallel execution, conflict detection, and result aggregation.

In [ ]:
"""
Listing 8-2: Distributed Multi-Agent Coordination System

Implements a ServiceRegistry, MessageBus, task decomposition, parallel
simulated agent execution, conflict detection and resolution, and
result aggregation via a LangGraph workflow.
"""

from typing import TypedDict, Annotated, List, Dict, Any, Optional
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import json, time, uuid, random


# ------------------------------------------------------------------ #
# Infrastructure Classes                                              #
# ------------------------------------------------------------------ #

class ServiceRegistry:
    """Agent discovery via capability indexing.
    In production, back with etcd, Consul, or a database."""

    def __init__(self):
        self.agents: Dict[str, dict] = {}

    def register(self, agent_id: str, name: str, capabilities: list,
                 proficiency: dict = None) -> str:
        """Register an agent with its capabilities."""
        self.agents[agent_id] = {
            "agent_id": agent_id,
            "name": name,
            "capabilities": capabilities,
            "proficiency": proficiency or {c: 0.8 for c in capabilities},
            "status": "healthy",
            "registered_at": time.time(),
            "last_heartbeat": time.time(),
            "current_load": 0.0,
        }
        return agent_id

    def discover(self, capability: str, min_proficiency: float = 0.0) -> list:
        """Find agents with a specific capability, ranked by proficiency."""
        matches = []
        for agent in self.agents.values():
            if capability in agent["capabilities"]:
                prof = agent["proficiency"].get(capability, 0)
                if prof >= min_proficiency:
                    matches.append({**agent, "match_proficiency": prof})
        matches.sort(key=lambda a: a["match_proficiency"], reverse=True)
        return matches

    def health_check(self) -> dict:
        """Return health status of all registered agents."""
        healthy = sum(1 for a in self.agents.values() if a["status"] == "healthy")
        return {
            "total_agents": len(self.agents),
            "healthy": healthy,
            "unhealthy": len(self.agents) - healthy,
            "agents": {
                aid: {"status": a["status"], "load": a["current_load"]}
                for aid, a in self.agents.items()
            },
        }


class MessageBus:
    """Simple message bus for inter-agent communication.
    In production, use Kafka, RabbitMQ, or Redis Streams."""

    def __init__(self):
        self.channels: Dict[str, list] = {}
        self.direct_messages: Dict[str, list] = {}

    def publish(self, channel: str, message: dict) -> None:
        """Publish a message to a topic channel."""
        if channel not in self.channels:
            self.channels[channel] = []
        self.channels[channel].append({
            "timestamp": time.time(),
            "message": message,
        })

    def subscribe(self, channel: str) -> list:
        """Get all messages from a channel."""
        return self.channels.get(channel, [])

    def send_direct(self, agent_id: str, message: dict) -> None:
        """Send a direct message to a specific agent."""
        if agent_id not in self.direct_messages:
            self.direct_messages[agent_id] = []
        self.direct_messages[agent_id].append(message)

    def get_messages(self, agent_id: str) -> list:
        """Retrieve direct messages for an agent."""
        msgs = self.direct_messages.get(agent_id, [])
        self.direct_messages[agent_id] = []  # Clear after reading
        return msgs


# ------------------------------------------------------------------ #
# State                                                               #
# ------------------------------------------------------------------ #

class CoordinationState(TypedDict):
    """State for distributed multi-agent coordination."""
    messages: Annotated[list[BaseMessage], add_messages]
    task: str
    subtasks: list[dict]
    agent_assignments: dict
    results: dict
    conflicts: list[dict]
    final_result: dict


# ------------------------------------------------------------------ #
# Shared Infrastructure Instances                                     #
# ------------------------------------------------------------------ #

registry = ServiceRegistry()
bus = MessageBus()

# Register specialist agents
registry.register("analyst-1", "Data Analyst", ["data_analysis", "statistics"],
                   {"data_analysis": 0.95, "statistics": 0.88})
registry.register("researcher-1", "Literature Researcher", ["literature_review", "summarization"],
                   {"literature_review": 0.92, "summarization": 0.85})
registry.register("writer-1", "Technical Writer", ["report_writing", "summarization"],
                   {"report_writing": 0.90, "summarization": 0.93})


# ------------------------------------------------------------------ #
# Simulated Agent Execution                                           #
# ------------------------------------------------------------------ #

def simulate_agent_execution(agent_id: str, subtask: dict) -> dict:
    """Simulate an agent executing a subtask and returning results."""
    agent = registry.agents.get(agent_id, {})
    agent_name = agent.get("name", agent_id)

    # Generate mock results based on subtask type
    results_db = {
        "data_analysis": {
            "findings": [
                "Dataset contains 10,000 samples across 5 categories",
                "Strong correlation (r=0.87) between variables X and Y",
                "Outlier cluster identified in 3% of samples",
            ],
            "confidence": 0.92,
            "methodology": "Statistical analysis with cross-validation",
        },
        "literature_review": {
            "findings": [
                "47 relevant papers identified (2020-2024)",
                "Three major research themes emerging",
                "Consensus on methodology A, disagreement on methodology B",
            ],
            "confidence": 0.88,
            "methodology": "Systematic review with keyword search",
        },
        "report_writing": {
            "findings": [
                "Executive summary drafted (500 words)",
                "Technical sections organized by theme",
                "Recommendations section includes 5 actionable items",
            ],
            "confidence": 0.90,
            "methodology": "Structured report template with citations",
        },
    }

    subtask_type = subtask.get("type", "general")
    mock = results_db.get(subtask_type, {
        "findings": [f"Completed {subtask_type} analysis"],
        "confidence": 0.75,
        "methodology": "General analysis",
    })

    return {
        "agent_id": agent_id,
        "agent_name": agent_name,
        "subtask_id": subtask.get("id", "unknown"),
        "status": "completed",
        **mock,
    }


# ------------------------------------------------------------------ #
# Node Functions                                                      #
# ------------------------------------------------------------------ #

def decompose_task(state: CoordinationState) -> dict:
    """Decompose the main task into subtasks."""
    task = state["task"]
    subtasks = [
        {
            "id": "subtask-1",
            "type": "data_analysis",
            "description": f"Analyze available data related to: {task}",
            "required_capability": "data_analysis",
            "priority": 1,
        },
        {
            "id": "subtask-2",
            "type": "literature_review",
            "description": f"Review literature on: {task}",
            "required_capability": "literature_review",
            "priority": 1,
        },
        {
            "id": "subtask-3",
            "type": "report_writing",
            "description": f"Write comprehensive report on: {task}",
            "required_capability": "report_writing",
            "priority": 2,
        },
    ]
    return {
        "subtasks": subtasks,
        "messages": [AIMessage(content=f"Decomposed task into {len(subtasks)} subtasks.")],
    }


def assign_agents(state: CoordinationState) -> dict:
    """Match subtasks to the best available agents."""
    assignments = {}
    for subtask in state["subtasks"]:
        capability = subtask["required_capability"]
        candidates = registry.discover(capability, min_proficiency=0.7)
        if candidates:
            best = candidates[0]
            assignments[subtask["id"]] = {
                "agent_id": best["agent_id"],
                "agent_name": best["name"],
                "proficiency": best["match_proficiency"],
            }
            bus.send_direct(best["agent_id"], {
                "type": "task_assignment",
                "subtask": subtask,
            })
    return {
        "agent_assignments": assignments,
        "messages": [AIMessage(content=(
            f"Assigned {len(assignments)} subtasks to agents. "
            f"Registry health: {json.dumps(registry.health_check())}"
        ))],
    }


def execute_parallel(state: CoordinationState) -> dict:
    """Execute all subtasks in parallel (simulated)."""
    results = {}
    for subtask in state["subtasks"]:
        assignment = state["agent_assignments"].get(subtask["id"], {})
        agent_id = assignment.get("agent_id")
        if agent_id:
            result = simulate_agent_execution(agent_id, subtask)
            results[subtask["id"]] = result
            bus.publish("results", {
                "subtask_id": subtask["id"],
                "agent_id": agent_id,
                "status": "completed",
            })
    return {
        "results": results,
        "messages": [AIMessage(content=f"Executed {len(results)} subtasks in parallel.")],
    }


def detect_conflicts(state: CoordinationState) -> dict:
    """Check results for contradictions between agents."""
    conflicts = []
    results_list = list(state["results"].values())
    for i in range(len(results_list)):
        for j in range(i + 1, len(results_list)):
            # Simulated conflict detection
            if random.random() < 0.3:  # 30% chance of detecting a conflict
                conflicts.append({
                    "type": "semantic",
                    "agents": [results_list[i]["agent_id"], results_list[j]["agent_id"]],
                    "description": (
                        f"Potential disagreement between {results_list[i]['agent_name']} "
                        f"and {results_list[j]['agent_name']} on methodology assessment"
                    ),
                    "severity": random.choice(["low", "medium"]),
                })
    return {
        "conflicts": conflicts,
        "messages": [AIMessage(content=(
            f"Conflict detection: {len(conflicts)} potential conflicts found."
        ))],
    }


def has_conflicts(state: CoordinationState) -> str:
    """Route based on whether conflicts were detected."""
    if state.get("conflicts"):
        return "resolve"
    return "aggregate"


def resolve_conflicts(state: CoordinationState) -> dict:
    """Resolve detected conflicts (simulated LLM arbitration)."""
    resolutions = []
    for conflict in state["conflicts"]:
        resolutions.append({
            "conflict": conflict["description"],
            "resolution": (
                f"After analysis, the methodological disagreement is resolved by "
                f"adopting a combined approach that incorporates strengths from both agents."
            ),
            "method": "LLM-based semantic arbitration",
        })
    return {
        "conflicts": [],  # Clear conflicts after resolution
        "messages": [AIMessage(content=f"Resolved {len(resolutions)} conflicts.")],
    }


def aggregate_results(state: CoordinationState) -> dict:
    """Aggregate all results into a final output."""
    all_findings = []
    for result in state["results"].values():
        all_findings.extend(result.get("findings", []))

    avg_confidence = (
        sum(r.get("confidence", 0) for r in state["results"].values())
        / max(len(state["results"]), 1)
    )

    final = {
        "task": state["task"],
        "agents_involved": len(state["agent_assignments"]),
        "subtasks_completed": len(state["results"]),
        "consolidated_findings": all_findings,
        "average_confidence": round(avg_confidence, 3),
        "conflicts_resolved": len(state.get("conflicts", [])) == 0,
        "recommendation": (
            "Analysis complete. High confidence in consolidated findings. "
            "Recommend proceeding with identified action items."
        ),
    }
    return {
        "final_result": final,
        "messages": [AIMessage(content="Results aggregated. Coordination complete.")],
    }


# ------------------------------------------------------------------ #
# Graph Construction                                                  #
# ------------------------------------------------------------------ #

def build_coordination_graph():
    """Build distributed multi-agent coordination workflow."""
    workflow = StateGraph(CoordinationState)
    workflow.add_node("decompose", decompose_task)
    workflow.add_node("assign", assign_agents)
    workflow.add_node("execute", execute_parallel)
    workflow.add_node("detect_conflicts", detect_conflicts)
    workflow.add_node("resolve_conflicts", resolve_conflicts)
    workflow.add_node("aggregate", aggregate_results)

    workflow.set_entry_point("decompose")
    workflow.add_edge("decompose", "assign")
    workflow.add_edge("assign", "execute")
    workflow.add_edge("execute", "detect_conflicts")
    workflow.add_conditional_edges(
        "detect_conflicts",
        has_conflicts,
        {"resolve": "resolve_conflicts", "aggregate": "aggregate"},
    )
    workflow.add_edge("resolve_conflicts", "aggregate")
    workflow.add_edge("aggregate", END)

    return workflow.compile()


# ------------------------------------------------------------------ #
# Usage Example                                                       #
# ------------------------------------------------------------------ #

graph = build_coordination_graph()

initial_state = {
    "messages": [HumanMessage(content="Analyze the impact of AI on materials discovery")],
    "task": "Comprehensive analysis of AI-driven materials discovery: current state, challenges, and opportunities",
    "subtasks": [],
    "agent_assignments": {},
    "results": {},
    "conflicts": [],
    "final_result": {},
}

result = graph.invoke(initial_state)

print("=" * 70)
print("DISTRIBUTED MULTI-AGENT COORDINATION RESULTS")
print("=" * 70)
final = result["final_result"]
print(f"\nTask: {final['task']}")
print(f"Agents involved: {final['agents_involved']}")
print(f"Subtasks completed: {final['subtasks_completed']}")
print(f"Average confidence: {final['average_confidence']}")
print(f"Conflicts resolved: {final['conflicts_resolved']}")
print(f"\nConsolidated findings:")
for f in final["consolidated_findings"]:
    print(f"  - {f}")
print(f"\nRecommendation: {final['recommendation']}")
print(f"\nRegistry health: {json.dumps(registry.health_check(), indent=2)}")


---

## Listing 8-3: Neurosymbolic Reasoning Agent

Combines symbolic forward-chaining inference with neural (LLM-based) reasoning. Routes queries to symbolic, neural, or hybrid paths based on the reasoning type required.

In [ ]:
"""
Listing 8-3: Neurosymbolic Reasoning Agent

Implements a SymbolicKnowledgeBase with forward-chaining inference,
classifies queries by reasoning type, and synthesizes symbolic and
neural results. All LLM calls are simulated.
"""

from typing import TypedDict, Annotated, List, Dict, Any, Optional, Tuple
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import json


# ------------------------------------------------------------------ #
# Symbolic Knowledge Base                                             #
# ------------------------------------------------------------------ #

class SymbolicKnowledgeBase:
    """Forward-chaining inference engine over facts and rules.
    In production, consider Prolog, Z3, or OWL/RDF reasoners."""

    def __init__(self):
        self.facts: set = set()
        self.rules: list = []

    def add_fact(self, fact: str) -> None:
        """Assert a fact into the knowledge base."""
        self.facts.add(fact.lower().strip())

    def add_rule(self, conditions: list, conclusion: str,
                 confidence: float = 1.0) -> None:
        """Add an inference rule: if all conditions hold, conclude."""
        self.rules.append({
            "conditions": [c.lower().strip() for c in conditions],
            "conclusion": conclusion.lower().strip(),
            "confidence": confidence,
        })

    def forward_chain(self, max_iterations: int = 100) -> list:
        """Apply all rules until no new facts are derived."""
        derivations = []
        for _ in range(max_iterations):
            new_facts = set()
            for rule in self.rules:
                if all(c in self.facts for c in rule["conditions"]):
                    if rule["conclusion"] not in self.facts:
                        new_facts.add(rule["conclusion"])
                        derivations.append({
                            "derived": rule["conclusion"],
                            "from_conditions": rule["conditions"],
                            "confidence": rule["confidence"],
                        })
            if not new_facts:
                break
            self.facts.update(new_facts)
        return derivations

    def query(self, proposition: str) -> Tuple[bool, list]:
        """Query whether a proposition is true, returning proof chain."""
        prop = proposition.lower().strip()
        if prop in self.facts:
            return True, [{"fact": prop, "source": "direct assertion"}]

        # Try forward chaining to derive
        derivations = self.forward_chain()
        if prop in self.facts:
            proof = [d for d in derivations if d["derived"] == prop]
            return True, proof if proof else [{"fact": prop, "source": "derived"}]

        return False, []

    def get_all_facts(self) -> list:
        """Return all known facts."""
        return sorted(self.facts)


# ------------------------------------------------------------------ #
# State                                                               #
# ------------------------------------------------------------------ #

class NeurosymbolicState(TypedDict):
    """State for neurosymbolic reasoning workflow."""
    messages: Annotated[list[BaseMessage], add_messages]
    query: str
    reasoning_type: str        # formal, probabilistic, creative
    symbolic_facts: list[str]
    symbolic_rules: list[dict]
    symbolic_results: dict
    neural_results: dict
    synthesis: dict


# ------------------------------------------------------------------ #
# Node Functions                                                      #
# ------------------------------------------------------------------ #

def classify_type(state: NeurosymbolicState) -> dict:
    """Classify the query to determine reasoning mode."""
    query = state["query"].lower()

    # Simple heuristic classification (would use LLM in production)
    formal_keywords = ["must", "required", "compliant", "regulation",
                       "rule", "if", "then", "always", "never", "prohibited"]
    creative_keywords = ["suggest", "brainstorm", "imagine", "what if",
                         "creative", "novel", "innovative"]

    formal_score = sum(1 for k in formal_keywords if k in query)
    creative_score = sum(1 for k in creative_keywords if k in query)

    if formal_score >= 2:
        reasoning_type = "formal"
    elif creative_score >= 2:
        reasoning_type = "creative"
    else:
        reasoning_type = "probabilistic"

    return {
        "reasoning_type": reasoning_type,
        "messages": [AIMessage(content=(
            f"Query classified as '{reasoning_type}' reasoning. "
            f"(formal_score={formal_score}, creative_score={creative_score})"
        ))],
    }


def symbolic_reason(state: NeurosymbolicState) -> dict:
    """Apply symbolic reasoning using the knowledge base."""
    kb = SymbolicKnowledgeBase()

    # Load facts and rules
    for fact in state.get("symbolic_facts", []):
        kb.add_fact(fact)
    for rule in state.get("symbolic_rules", []):
        kb.add_rule(
            conditions=rule["conditions"],
            conclusion=rule["conclusion"],
            confidence=rule.get("confidence", 1.0),
        )

    # Forward chain to derive new facts
    derivations = kb.forward_chain()

    # Check specific propositions from the query
    query_words = state["query"].lower().split()
    checked_propositions = []
    for fact in kb.get_all_facts():
        if any(w in fact for w in query_words if len(w) > 3):
            checked_propositions.append(fact)

    return {
        "symbolic_results": {
            "derivations": derivations,
            "total_facts": len(kb.get_all_facts()),
            "all_facts": kb.get_all_facts(),
            "relevant_facts": checked_propositions,
            "reasoning_complete": True,
        },
        "messages": [AIMessage(content=(
            f"Symbolic reasoning: derived {len(derivations)} new facts. "
            f"Total knowledge base: {len(kb.get_all_facts())} facts."
        ))],
    }


def neural_reason(state: NeurosymbolicState) -> dict:
    """Apply neural (LLM-based) reasoning (simulated)."""
    query = state["query"]

    # Simulated LLM response
    neural_output = {
        "interpretation": (
            f"The query '{query[:60]}...' requires analysis of contextual factors "
            "and domain knowledge beyond formal logic."
        ),
        "reasoning_steps": [
            "Step 1: Identify key concepts and relationships in the query",
            "Step 2: Apply domain knowledge to interpret ambiguous terms",
            "Step 3: Consider edge cases and exceptions not captured by rules",
            "Step 4: Provide probabilistic assessment with confidence intervals",
        ],
        "conclusion": (
            "Based on contextual analysis, the query involves both rule-based "
            "and judgment-based components. Formal rules provide the foundation, "
            "but practical application requires domain expertise."
        ),
        "confidence": 0.82,
        "caveats": [
            "LLM reasoning is probabilistic, not guaranteed",
            "Novel scenarios may not match training distribution",
        ],
    }

    return {
        "neural_results": neural_output,
        "messages": [AIMessage(content=(
            f"Neural reasoning complete. Confidence: {neural_output['confidence']}"
        ))],
    }


def synthesize(state: NeurosymbolicState) -> dict:
    """Synthesize symbolic and neural reasoning results."""
    symbolic = state.get("symbolic_results", {})
    neural = state.get("neural_results", {})

    # Combine results, leading with formally verified conclusions
    formally_proven = symbolic.get("derivations", [])
    neural_insights = neural.get("reasoning_steps", [])

    synthesis = {
        "query": state["query"],
        "reasoning_type": state["reasoning_type"],
        "formally_proven_conclusions": [
            {
                "conclusion": d["derived"],
                "confidence": d["confidence"],
                "proof": f"Derived from: {', '.join(d['from_conditions'])}",
                "verification": "FORMALLY VERIFIED",
            }
            for d in formally_proven
        ],
        "probabilistic_insights": {
            "neural_conclusion": neural.get("conclusion", ""),
            "confidence": neural.get("confidence", 0),
            "verification": "PROBABILISTIC (LLM-based)",
        },
        "combined_assessment": (
            f"Analysis used {state['reasoning_type']} reasoning. "
            f"{len(formally_proven)} conclusions are formally verified. "
            f"Neural reasoning provides additional context with "
            f"{neural.get('confidence', 0):.0%} confidence."
        ),
        "recommendation": (
            "Trust formally verified conclusions for compliance decisions. "
            "Use neural insights for context and edge cases."
        ),
    }

    return {
        "synthesis": synthesis,
        "messages": [AIMessage(content=(
            f"Synthesis complete. {len(formally_proven)} formal proofs + "
            f"neural context at {neural.get('confidence', 0):.0%} confidence."
        ))],
    }


def route_reasoning(state: NeurosymbolicState) -> str:
    """Route to appropriate reasoning path."""
    rt = state.get("reasoning_type", "probabilistic")
    if rt == "formal":
        return "symbolic"
    elif rt == "creative":
        return "neural"
    return "symbolic"  # hybrid: start with symbolic


# ------------------------------------------------------------------ #
# Graph Construction                                                  #
# ------------------------------------------------------------------ #

def build_neurosymbolic_agent():
    """Build neurosymbolic reasoning workflow."""
    workflow = StateGraph(NeurosymbolicState)
    workflow.add_node("classify", classify_type)
    workflow.add_node("symbolic", symbolic_reason)
    workflow.add_node("neural", neural_reason)
    workflow.add_node("synthesize", synthesize)

    workflow.set_entry_point("classify")
    workflow.add_conditional_edges(
        "classify",
        route_reasoning,
        {"symbolic": "symbolic", "neural": "neural"},
    )
    workflow.add_edge("symbolic", "neural")
    workflow.add_edge("neural", "synthesize")
    workflow.add_edge("synthesize", END)

    return workflow.compile()


# ------------------------------------------------------------------ #
# Usage Example                                                       #
# ------------------------------------------------------------------ #

graph = build_neurosymbolic_agent()

# Regulatory compliance question demonstrating symbolic logic advantage
initial_state = {
    "messages": [HumanMessage(content="Is this chemical process compliant with safety regulations?")],
    "query": (
        "A chemical process must be compliant with regulation X. "
        "The process uses reagent A at 200C. Regulation X requires temperature "
        "below 300C and prohibited substances must never be used. "
        "Is this process compliant if reagent A is not prohibited?"
    ),
    "reasoning_type": "",
    "symbolic_facts": [
        "process uses reagent a",
        "process temperature is 200c",
        "regulation x requires temperature below 300c",
        "reagent a is not prohibited",
        "process temperature is below 300c",
    ],
    "symbolic_rules": [
        {
            "conditions": [
                "process temperature is below 300c",
                "reagent a is not prohibited",
            ],
            "conclusion": "process is temperature compliant",
            "confidence": 1.0,
        },
        {
            "conditions": [
                "process is temperature compliant",
                "reagent a is not prohibited",
            ],
            "conclusion": "process is fully compliant with regulation x",
            "confidence": 1.0,
        },
    ],
    "symbolic_results": {},
    "neural_results": {},
    "synthesis": {},
}

result = graph.invoke(initial_state)

print("=" * 70)
print("NEUROSYMBOLIC REASONING RESULTS")
print("=" * 70)
syn = result["synthesis"]
print(f"\nQuery: {syn['query'][:100]}...")
print(f"Reasoning type: {syn['reasoning_type']}")
print(f"\nFormally proven conclusions:")
for c in syn["formally_proven_conclusions"]:
    print(f"  [{c['verification']}] {c['conclusion']} (confidence: {c['confidence']})")
    print(f"    Proof: {c['proof']}")
print(f"\nNeural assessment:")
print(f"  {syn['probabilistic_insights']['neural_conclusion'][:150]}...")
print(f"  Confidence: {syn['probabilistic_insights']['confidence']}")
print(f"\nCombined: {syn['combined_assessment']}")
print(f"Recommendation: {syn['recommendation']}")


---

## Listing 8-4: Agent Marketplace Registration and Discovery System

A complete agent marketplace with registration, multi-dimensional search, a certification pipeline (functional tests, safety evaluation, performance profiling), and recommendation via a LangGraph workflow.

In [ ]:
"""
Listing 8-4: Agent Marketplace Registration and Discovery System

Implements AgentListing, AgentMarketplace with search and metrics,
CertificationPipeline, and a LangGraph workflow for search -> evaluate
-> recommend. All evaluations are simulated.
"""

from typing import TypedDict, Annotated, List, Dict, Any, Optional
from dataclasses import dataclass, field
from enum import Enum
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import json, uuid, time, random


# ------------------------------------------------------------------ #
# Data Classes                                                        #
# ------------------------------------------------------------------ #

class CertificationLevel(str, Enum):
    UNCERTIFIED = "uncertified"
    BASIC = "basic"
    STANDARD = "standard"
    ENTERPRISE = "enterprise"


@dataclass
class AgentListing:
    """Marketplace listing with all metadata for discovery and deployment."""
    agent_id: str
    name: str
    description: str
    capabilities: list = field(default_factory=list)
    domains: list = field(default_factory=list)
    pricing_model: dict = field(default_factory=lambda: {"model": "per_request", "price_usd": 0.01})
    performance_metrics: dict = field(default_factory=dict)
    certified: bool = False
    certification_level: CertificationLevel = CertificationLevel.UNCERTIFIED
    version: str = "1.0"
    downloads: int = 0
    rating: float = 0.0
    rating_count: int = 0

    def to_agent_card(self) -> dict:
        """Generate A2A-compatible agent card for protocol-level discovery."""
        return {
            "name": self.name,
            "description": self.description,
            "capabilities": self.capabilities,
            "domains": self.domains,
            "certification": self.certification_level.value,
            "version": self.version,
            "pricing": self.pricing_model,
        }


@dataclass
class CertificationResult:
    """Result of an agent certification evaluation."""
    passed: bool
    functional_score: float
    safety_score: float
    performance_score: float
    level: CertificationLevel
    report: str


# ------------------------------------------------------------------ #
# Agent Marketplace                                                   #
# ------------------------------------------------------------------ #

class AgentMarketplace:
    """Registry with multi-dimensional indexing for efficient discovery."""

    def __init__(self):
        self.listings: Dict[str, AgentListing] = {}
        self._capability_index: Dict[str, list] = {}
        self._domain_index: Dict[str, list] = {}

    def register(self, listing: AgentListing) -> str:
        """Register an agent listing and update indexes."""
        self.listings[listing.agent_id] = listing
        for cap in listing.capabilities:
            self._capability_index.setdefault(cap, []).append(listing.agent_id)
        for domain in listing.domains:
            self._domain_index.setdefault(domain, []).append(listing.agent_id)
        return listing.agent_id

    def search(self, query: str = None, capability: str = None,
               domain: str = None, min_certification: CertificationLevel = None,
               min_rating: float = 0.0) -> list:
        """Search by capability, domain, text, certification, and rating."""
        candidates = set(self.listings.keys())

        if capability:
            cap_matches = set(self._capability_index.get(capability, []))
            candidates &= cap_matches

        if domain:
            dom_matches = set(self._domain_index.get(domain, []))
            candidates &= dom_matches

        results = []
        for aid in candidates:
            listing = self.listings[aid]
            if min_rating and listing.rating < min_rating:
                continue
            if min_certification:
                cert_order = list(CertificationLevel)
                if cert_order.index(listing.certification_level) < cert_order.index(min_certification):
                    continue
            if query:
                q = query.lower()
                if not (q in listing.name.lower() or q in listing.description.lower()
                        or any(q in c.lower() for c in listing.capabilities)):
                    continue
            results.append(listing)

        # Rank by certification level, then rating, then downloads
        cert_order = list(CertificationLevel)
        results.sort(key=lambda l: (
            cert_order.index(l.certification_level),
            l.rating,
            l.downloads,
        ), reverse=True)
        return results

    def get_metrics(self, agent_id: str) -> dict:
        """Get usage metrics for an agent."""
        listing = self.listings.get(agent_id)
        if not listing:
            return {"error": "Agent not found"}
        return {
            "agent_id": agent_id,
            "name": listing.name,
            "downloads": listing.downloads,
            "rating": listing.rating,
            "rating_count": listing.rating_count,
            "certification": listing.certification_level.value,
            "performance_metrics": listing.performance_metrics,
        }


# ------------------------------------------------------------------ #
# Certification Pipeline                                              #
# ------------------------------------------------------------------ #

class CertificationPipeline:
    """Evaluates agents across functional, safety, and performance dimensions."""

    def __init__(self, marketplace: AgentMarketplace):
        self.marketplace = marketplace

    def run_functional_tests(self, agent_id: str) -> dict:
        """Simulated functional testing."""
        return {
            "agent_id": agent_id,
            "test_cases_run": 50,
            "passed": random.randint(42, 50),
            "failed": 0,  # Will be computed
            "score": round(random.uniform(0.85, 0.99), 3),
            "categories": {
                "input_handling": round(random.uniform(0.9, 1.0), 2),
                "output_quality": round(random.uniform(0.85, 0.98), 2),
                "error_handling": round(random.uniform(0.8, 0.95), 2),
                "edge_cases": round(random.uniform(0.75, 0.95), 2),
            },
        }

    def run_safety_eval(self, agent_id: str) -> dict:
        """Simulated safety evaluation."""
        return {
            "agent_id": agent_id,
            "safety_score": round(random.uniform(0.88, 0.99), 3),
            "categories": {
                "prompt_injection_resistance": round(random.uniform(0.9, 1.0), 2),
                "data_privacy": round(random.uniform(0.92, 1.0), 2),
                "output_safety": round(random.uniform(0.88, 0.99), 2),
                "behavioral_bounds": round(random.uniform(0.85, 0.98), 2),
            },
            "vulnerabilities_found": random.randint(0, 2),
            "critical_issues": 0,
        }

    def run_performance_profile(self, agent_id: str) -> dict:
        """Simulated performance profiling."""
        return {
            "agent_id": agent_id,
            "performance_score": round(random.uniform(0.80, 0.98), 3),
            "latency_p50_ms": random.randint(100, 500),
            "latency_p99_ms": random.randint(500, 2000),
            "throughput_rps": round(random.uniform(5, 50), 1),
            "memory_mb": random.randint(256, 1024),
            "cost_per_1k_requests_usd": round(random.uniform(0.5, 10.0), 2),
        }

    def certify(self, agent_id: str) -> CertificationResult:
        """Run full certification pipeline and determine level."""
        functional = self.run_functional_tests(agent_id)
        safety = self.run_safety_eval(agent_id)
        performance = self.run_performance_profile(agent_id)

        func_score = functional["score"]
        safe_score = safety["safety_score"]
        perf_score = performance["performance_score"]

        # Determine certification level based on scores
        if func_score >= 0.95 and safe_score >= 0.95 and perf_score >= 0.90:
            level = CertificationLevel.ENTERPRISE
        elif func_score >= 0.90 and safe_score >= 0.90:
            level = CertificationLevel.STANDARD
        elif func_score >= 0.80 and safe_score >= 0.85:
            level = CertificationLevel.BASIC
        else:
            level = CertificationLevel.UNCERTIFIED

        passed = level != CertificationLevel.UNCERTIFIED

        # Update marketplace listing
        if agent_id in self.marketplace.listings:
            listing = self.marketplace.listings[agent_id]
            listing.certified = passed
            listing.certification_level = level
            listing.performance_metrics = {
                "functional_score": func_score,
                "safety_score": safe_score,
                "performance_score": perf_score,
            }

        report = (
            f"Certification {'PASSED' if passed else 'FAILED'} at level: {level.value}. "
            f"Functional: {func_score:.3f}, Safety: {safe_score:.3f}, "
            f"Performance: {perf_score:.3f}."
        )

        return CertificationResult(
            passed=passed,
            functional_score=func_score,
            safety_score=safe_score,
            performance_score=perf_score,
            level=level,
            report=report,
        )


# ------------------------------------------------------------------ #
# State & Workflow                                                    #
# ------------------------------------------------------------------ #

class MarketplaceState(TypedDict):
    """State for marketplace workflow."""
    messages: Annotated[list[BaseMessage], add_messages]
    user_requirements: dict
    search_results: list[dict]
    certification_results: list[dict]
    recommendation: dict


# Shared marketplace instance
marketplace = AgentMarketplace()
pipeline = CertificationPipeline(marketplace)

# Register sample agents
sample_agents = [
    AgentListing(
        agent_id="agent-data-001",
        name="DataAnalyzer Pro",
        description="Advanced data analysis agent with statistical modeling capabilities",
        capabilities=["data_analysis", "statistics", "visualization", "reporting"],
        domains=["finance", "research", "healthcare"],
        pricing_model={"model": "per_request", "price_usd": 0.05},
        version="2.1",
        downloads=1500,
        rating=4.5,
        rating_count=230,
    ),
    AgentListing(
        agent_id="agent-research-002",
        name="ResearchAssistant AI",
        description="Literature review and research synthesis agent",
        capabilities=["literature_review", "summarization", "citation_management"],
        domains=["research", "academia", "healthcare"],
        pricing_model={"model": "subscription", "price_usd": 29.99},
        version="1.5",
        downloads=800,
        rating=4.2,
        rating_count=120,
    ),
    AgentListing(
        agent_id="agent-code-003",
        name="CodeReview Bot",
        description="Automated code review and quality assurance agent",
        capabilities=["code_review", "testing", "documentation", "refactoring"],
        domains=["software", "devops"],
        pricing_model={"model": "per_request", "price_usd": 0.02},
        version="3.0",
        downloads=3200,
        rating=4.7,
        rating_count=450,
    ),
]

for agent in sample_agents:
    marketplace.register(agent)


# ------------------------------------------------------------------ #
# Node Functions                                                      #
# ------------------------------------------------------------------ #

def search_node(state: MarketplaceState) -> dict:
    """Search marketplace based on user requirements."""
    reqs = state["user_requirements"]
    results = marketplace.search(
        query=reqs.get("query"),
        capability=reqs.get("capability"),
        domain=reqs.get("domain"),
    )
    return {
        "search_results": [r.to_agent_card() for r in results],
        "messages": [AIMessage(content=f"Found {len(results)} agents matching requirements.")],
    }


def evaluate_node(state: MarketplaceState) -> dict:
    """Certify/evaluate top search results."""
    cert_results = []
    # Get agent IDs from search results
    for card in state["search_results"][:3]:  # Evaluate top 3
        # Find agent_id by name
        for aid, listing in marketplace.listings.items():
            if listing.name == card["name"]:
                cert = pipeline.certify(aid)
                cert_results.append({
                    "agent_id": aid,
                    "name": card["name"],
                    "passed": cert.passed,
                    "level": cert.level.value,
                    "functional_score": cert.functional_score,
                    "safety_score": cert.safety_score,
                    "performance_score": cert.performance_score,
                    "report": cert.report,
                })
                break
    return {
        "certification_results": cert_results,
        "messages": [AIMessage(content=f"Evaluated {len(cert_results)} agents.")],
    }


def recommend_node(state: MarketplaceState) -> dict:
    """Generate recommendation based on search and certification results."""
    certified = [c for c in state["certification_results"] if c["passed"]]

    if not certified:
        recommendation = {
            "status": "no_qualified_agents",
            "message": "No agents passed certification for your requirements.",
            "alternatives": "Consider broadening search criteria.",
        }
    else:
        # Rank by composite score
        for c in certified:
            c["composite"] = round(
                0.4 * c["functional_score"]
                + 0.35 * c["safety_score"]
                + 0.25 * c["performance_score"], 3
            )
        certified.sort(key=lambda c: c["composite"], reverse=True)
        best = certified[0]

        recommendation = {
            "status": "recommendation_ready",
            "top_pick": {
                "name": best["name"],
                "agent_id": best["agent_id"],
                "certification_level": best["level"],
                "composite_score": best["composite"],
                "report": best["report"],
            },
            "alternatives": [
                {"name": c["name"], "score": c["composite"], "level": c["level"]}
                for c in certified[1:]
            ],
            "total_evaluated": len(state["certification_results"]),
            "total_certified": len(certified),
        }

    return {
        "recommendation": recommendation,
        "messages": [AIMessage(content="Recommendation generated.")],
    }


# ------------------------------------------------------------------ #
# Graph Construction                                                  #
# ------------------------------------------------------------------ #

def build_marketplace_graph():
    """Build search -> evaluate -> recommend workflow."""
    workflow = StateGraph(MarketplaceState)
    workflow.add_node("search", search_node)
    workflow.add_node("evaluate", evaluate_node)
    workflow.add_node("recommend", recommend_node)

    workflow.set_entry_point("search")
    workflow.add_edge("search", "evaluate")
    workflow.add_edge("evaluate", "recommend")
    workflow.add_edge("recommend", END)

    return workflow.compile()


# ------------------------------------------------------------------ #
# Usage Example                                                       #
# ------------------------------------------------------------------ #

graph = build_marketplace_graph()

initial_state = {
    "messages": [HumanMessage(content="Find an agent for research data analysis")],
    "user_requirements": {
        "query": "data analysis",
        "capability": "data_analysis",
        "domain": "research",
    },
    "search_results": [],
    "certification_results": [],
    "recommendation": {},
}

result = graph.invoke(initial_state)

print("=" * 70)
print("AGENT MARKETPLACE RESULTS")
print("=" * 70)
rec = result["recommendation"]
print(f"\nStatus: {rec['status']}")
if rec.get("top_pick"):
    top = rec["top_pick"]
    print(f"\nTop Pick: {top['name']}")
    print(f"  Agent ID: {top['agent_id']}")
    print(f"  Certification: {top['certification_level']}")
    print(f"  Composite score: {top['composite_score']}")
    print(f"  Report: {top['report']}")

if rec.get("alternatives"):
    print(f"\nAlternatives:")
    for alt in rec["alternatives"]:
        print(f"  - {alt['name']} (score: {alt['score']}, level: {alt['level']})")

print(f"\nTotal evaluated: {rec.get('total_evaluated', 0)}")
print(f"Total certified: {rec.get('total_certified', 0)}")

# Show marketplace state
print(f"\nMarketplace stats:")
print(f"  Total listings: {len(marketplace.listings)}")
for aid, listing in marketplace.listings.items():
    metrics = marketplace.get_metrics(aid)
    print(f"  - {listing.name}: cert={listing.certification_level.value}, "
          f"rating={listing.rating}, downloads={listing.downloads}")
